# Archaic (Shakespeare) – 01 Data

Load the Shakespeare corpus, clean it, split it 90/10, and cut it into 256-token blocks for training.
The blocks are saved to disk and loaded by `02_train_generation.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
DATA_PATH  = "data.txt"                         # path to the provided Shakespeare file
OUT_DIR    = Path("../../results/archaic")      # shared with 02_train_generation
BLOCK      = 256                                # tokens per training block
SEED       = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and clean

In [ ]:
raw = load_dataset("text", data_files={"train": DATA_PATH}, sample_by="paragraph")["train"]

# Rows start with a leading "\n" (where speaker labels were removed), so strip them,
# and drop anything empty.
ds = raw.map(lambda ex: {"text": ex["text"].strip()})
ds = ds.filter(lambda ex: len(ex["text"]) > 0)
print(len(raw), "->", len(ds), "rows")

print(ds[0]["text"][:500])

**Formatting notes**
- The provided data has **no speaker names and no stage directions**, so nothing had to be handled.
- Each row is one speech; line breaks inside a speech (verse lines) are kept as `\n`.
- Leading newlines are stripped and speeches are re-joined with `\n\n` before tokenizing.

## 3. Train / test split

In [ ]:
# Contiguous 90/10 split (no shuffle), so test is unseen material.
data = ds.train_test_split(train_size=0.9, shuffle=False)
data

## 4. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token, "| vocab:", tokenizer.vocab_size)

In [ ]:
# Quick look at how text becomes tokens.
example = data["train"][0]["text"][:200]
ids = tokenizer(example)["input_ids"]
pieces = tokenizer.convert_ids_to_tokens(ids)
print(list(zip(ids, pieces))[:20])

print("\\n ->", tokenizer("\n")["input_ids"], "| \\n\\n ->", tokenizer("\n\n")["input_ids"])
print("round trip ok:", tokenizer.decode(ids) == example)

The tokenizer is byte-level: `Ġ` is a space and `Ċ` is a newline in the raw pieces. `\n\n` is a single token, so speech boundaries are cheap.

## 5. Build 256-token blocks

In [ ]:
def make_blocks(ids, block=BLOCK):
    n = len(ids) // block                      # leftover tail is dropped
    return [ids[i * block:(i + 1) * block] for i in range(n)]

def to_ds(blocks):
    # labels = copy of input_ids; the model shifts them internally
    return Dataset.from_dict({"input_ids": blocks, "labels": [b.copy() for b in blocks]})

train_ids = tokenizer("\n\n".join(data["train"]["text"]))["input_ids"]
test_ids  = tokenizer("\n\n".join(data["test"]["text"]))["input_ids"]

train_blocks, test_blocks = make_blocks(train_ids), make_blocks(test_ids)
assert all(len(b) == BLOCK for b in train_blocks + test_blocks)

lm_data = DatasetDict({
    "train": to_ds(train_blocks).shuffle(seed=SEED),   # shuffle train only
    "test":  to_ds(test_blocks),
})
print(f"train: {len(train_ids):,} tokens -> {len(train_blocks)} blocks")
print(f"test:  {len(test_ids):,} tokens -> {len(test_blocks)} blocks")

In [ ]:
print(tokenizer.decode(lm_data["train"][0]["input_ids"])[:600])

## 6. Save

In [ ]:
lm_data.save_to_disk(OUT_DIR / "blocks")
print("saved to", OUT_DIR / "blocks")